Cuu Duong Dang & Thi My Dung Le (2025)

In [ ]:
import os
import numpy as np
import pandas as pd
import random
import tensorflow as tf

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler, FunctionTransformer
from sklearn.decomposition import PCA
from sklearn.metrics import (
    classification_report, precision_score, recall_score, f1_score, roc_auc_score,
    balanced_accuracy_score, confusion_matrix, ConfusionMatrixDisplay
)
from sklearn.ensemble import AdaBoostClassifier, StackingClassifier
from xgboost import XGBClassifier
from sklearn.linear_model import LogisticRegression

import matplotlib.pyplot as plt
import joblib
from imblearn.over_sampling import SMOTENC

seed_value = 50
np.random.seed(seed_value)
random.seed(seed_value)
tf.random.set_seed(seed_value)

data = pd.read_csv('diabetes_prediction_dataset.csv')
data.dropna(inplace=True)
data['gender'] = data['gender'].map({'Male': 1, 'Female': 0})
data = pd.get_dummies(data, columns=['smoking_history'], drop_first=True)
data.fillna(0, inplace=True)

scaler = None
pca = None

def data_preprocessing(X_df, y_df, fit):
    # Log1p
    log_transformer = FunctionTransformer(np.log1p, validate=True)
    X_df = log_transformer.transform(X_df)
    X_df = pd.DataFrame(X_df)
    X_df.fillna(0, inplace=True)

    # Scale
    global scaler
    if fit:
        scaler = MinMaxScaler()
        X_df = scaler.fit_transform(X_df)
    else:
        X_df = scaler.transform(X_df)

    # PCA 99%
    global pca
    if fit:
        pca = PCA(n_components=0.99)
        X_df = pca.fit_transform(X_df)
    else:
        X_df = pca.transform(X_df)

    return X_df, y_df

X = data.drop(['diabetes'], axis=1).astype('float32')
y = data['diabetes']

X_train_full, X_temp, y_train_full, y_temp = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, random_state=42, stratify=y_temp
)

print(f"Original X shape: {X.shape}")
print(f"Train set shape: {X_train_full.shape}")
print(f"Validation set shape: {X_val.shape}")
print(f"Test set shape: {X_test.shape}")


sm = SMOTENC(categorical_features=[X_train_full.columns.get_loc('gender')], random_state=42)
X_train_smote, y_train_smote = sm.fit_resample(X_train_full, y_train_full)
print("Train size after SMOTENC:", X_train_smote.shape[0])

X_train_processed, y_train_processed = data_preprocessing(X_train_smote, y_train_smote, fit=True)
X_val_processed, _ = data_preprocessing(X_val, None, fit=False)
X_test_processed, _ = data_preprocessing(X_test, None, fit=False) 

X_train, y_train = X_train_processed, y_train_processed
X_val, y_val = X_val_processed, y_val
X_test, y_test = X_test_processed, y_test

def build_stacking_model(random_state=42):
    base_estimators = [
        ('xgb', XGBClassifier(random_state=random_state)),
        ('ada', AdaBoostClassifier(random_state=random_state))
    ]
    final_estimator = LogisticRegression(random_state=random_state, max_iter=1000)

    stacking_model = StackingClassifier(
        estimators=base_estimators,
        final_estimator=final_estimator,
        cv=5,
        passthrough=True
    )
    return stacking_model

stack_model = build_stacking_model(random_state=42)
stack_model.fit(X_train, y_train)

y_pred_val = stack_model.predict(X_val)
y_proba_val = stack_model.predict_proba(X_val)[:, 1] 

y_pred_test = stack_model.predict(X_test)
y_proba_test = stack_model.predict_proba(X_test)[:, 1]


print(classification_report(y_test, y_pred_test))
print(f"Precision: {precision_score(y_test, y_pred_test):.5f}")
print(f"Recall: {recall_score(y_test, y_pred_test):.5f}")
print(f"F1 Score: {f1_score(y_test, y_pred_test):.5f}")
print(f"Accuracy: {stack_model.score(X_test, y_test):.5f}")
balanced_acc_test = balanced_accuracy_score(y_test, y_pred_test)
print(f"Balanced Accuracy: {balanced_acc_test:.5f}")
print(f"ROC AUC Score: {roc_auc_score(y_test, y_proba_test):.5f}")


Original X shape: (100000, 12)
Train set shape: (70000, 12)
Validation set shape: (15000, 12)
Test set shape: (15000, 12)
Train size after SMOTENC: 128100


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2732: UserWarning: X has feature names, but FunctionTransformer was fitted without feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2732: UserWarning: X has feature names, but FunctionTransformer was fitted without feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2732: UserWarning: X has feature names, but FunctionTransformer was fitted without feature names
  warnings.warn(



Stacking Ensemble Classification Report (Test Set):
              precision    recall  f1-score   support

           0       0.98      0.97      0.97     13725
           1       0.68      0.78      0.73      1275

    accuracy                           0.95     15000
   macro avg       0.83      0.87      0.85     15000
weighted avg       0.95      0.95      0.95     15000

Precision: 0.68320
Recall: 0.77804
F1 Score: 0.72754
Accuracy: 0.95047
Balanced Accuracy: 0.87226
ROC AUC Score: 0.96802


Haneen Altartouri (2026)

In [ ]:
import pandas as pd
import numpy as np

from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, classification_report

from sklearn.mixture import GaussianMixture
from sklearn.neighbors import KernelDensity
from preprocessing import clean_and_encode
import warnings
warnings.filterwarnings('ignore')


df = pd.read_csv('diabetes_prediction_dataset.csv')
continuous_cols = ['age', 'bmi', 'blood_glucose_level', 'HbA1c_level']
results = clean_and_encode(df, df)

data = results["Dataset_1"]

X_train_full = data['X_train']
y_train = data['y_train']

X_test_full = data['X_test']
y_test = data['y_test']

# Lấy lại continuous từ data gốc (không phải encoded)
X_train_cont = df.loc[X_train_full.index, continuous_cols]
X_test_cont = df.loc[X_test_full.index, continuous_cols]

# CUSTOM TRANSFORMER
class GMMKDETransformer(BaseEstimator, TransformerMixin):
    def __init__(self, n_components=2, bandwidth=0.5):
        self.n_components = n_components
        self.bandwidth = bandwidth

    def fit(self, X, y=None):
        self.scaler = StandardScaler()
        X_scaled = self.scaler.fit_transform(X)

        self.gmm = GaussianMixture(n_components=self.n_components, random_state=42)
        self.gmm.fit(X_scaled)

        self.kde = KernelDensity(kernel='gaussian', bandwidth=self.bandwidth)
        self.kde.fit(X_scaled)

        return self

    def transform(self, X):
        X_scaled = self.scaler.transform(X)

        gmm_probs = self.gmm.predict_proba(X_scaled)
        kde_log = self.kde.score_samples(X_scaled).reshape(-1, 1)

        return np.hstack([X_scaled, gmm_probs, kde_log])

# BLOCK-WISE STACKING
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

meta_train_features = []
meta_test_features = []

for col in continuous_cols:
    print(f"Processing: {col}")

    X_tr_col = X_train_cont[[col]].values
    X_te_col = X_test_cont[[col]].values

    pipeline = Pipeline([
        ('gmm_kde', GMMKDETransformer()),
        ('rf', RandomForestClassifier(n_estimators=50, max_depth=5, random_state=42))
    ])

    oof_preds = cross_val_predict(
        pipeline,
        X_tr_col,
        y_train,
        cv=skf,
        method='predict_proba'
    )[:, 1]

    meta_train_features.append(oof_preds.reshape(-1, 1))

    pipeline.fit(X_tr_col, y_train)
    test_preds = pipeline.predict_proba(X_te_col)[:, 1]

    meta_test_features.append(test_preds.reshape(-1, 1))


# ADD ENCODED FEATURES (RẤT QUAN TRỌNG)
meta_train_features.append(X_train_full.values)
meta_test_features.append(X_test_full.values)

X_meta_train = np.hstack(meta_train_features)
X_meta_test = np.hstack(meta_test_features)

print("Meta shape:", X_meta_train.shape)


# META MODEL
meta_model = LogisticRegression(max_iter=1000)
meta_model.fit(X_meta_train, y_train)


y_pred = meta_model.predict(X_meta_test)

acc = accuracy_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)

print("\nKẾT QUẢ CUỐI")
print(f"Accuracy: {acc:.4f}")
print(f"F1-score: {f1:.4f}")
print(classification_report(y_test, y_pred))

Processing: age
Processing: bmi
Processing: blood_glucose_level
Processing: HbA1c_level
Meta shape: (67302, 17)

KẾT QUẢ CUỐI
Accuracy: 0.9606
F1-score: 0.7478
              precision    recall  f1-score   support

           0       0.97      0.99      0.98     13150
           1       0.86      0.66      0.75      1272

    accuracy                           0.96     14422
   macro avg       0.91      0.83      0.86     14422
weighted avg       0.96      0.96      0.96     14422



Shoyeb Mahmud (2024)

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import GradientBoostingClassifier
from imblearn.over_sampling import SMOTE
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report
from preprocessing import clean_and_encode


df = pd.read_csv('diabetes_prediction_dataset.csv')
clean_data = clean_and_encode(df, df)['Dataset_1']
X_train, y_train = clean_data['X_train'], clean_data['y_train']
X_val, y_val = clean_data['X_val'], clean_data['y_val']
X_test, y_test = clean_data['X_test'], clean_data['y_test']

smote = SMOTE(random_state=42)
X_train_smote, y_train_smote = smote.fit_resample(X_train, y_train)

print(f"Số lượng nhãn trước SMOTE: {y_train.value_counts().to_dict()}")
print(f"Số lượng nhãn sau SMOTE: {y_train_smote.value_counts().to_dict()}")

gbt_model = GradientBoostingClassifier(random_state=42)
gbt_model.fit(X_train_smote, y_train_smote)

y_pred = gbt_model.predict(X_test)


print(f"Accuracy:  {accuracy_score(y_test, y_pred):.4f}")
print(f"Precision: {precision_score(y_test, y_pred):.4f}")
print(f"Recall:    {recall_score(y_test, y_pred):.4f}")
print(f"F1 Score:  {f1_score(y_test, y_pred):.4f}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred))

Số lượng nhãn trước SMOTE: {0: 61365, 1: 5937}
Số lượng nhãn sau SMOTE: {0: 61365, 1: 61365}
=== KẾT QUẢ CỦA PHƯƠNG PHÁP SMOTE + GRADIENT BOOSTING ===
Accuracy:  0.9664
Precision: 0.8655
Recall:    0.7335
F1 Score:  0.7940

Classification Report:
              precision    recall  f1-score   support

           0       0.97      0.99      0.98     13150
           1       0.87      0.73      0.79      1272

    accuracy                           0.97     14422
   macro avg       0.92      0.86      0.89     14422
weighted avg       0.96      0.97      0.97     14422



Yassine Himeur (2024)

In [ ]:
from sklearn.ensemble import StackingClassifier, RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from xgboost import XGBClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score
from preprocessing import clean_and_encode


df = pd.read_csv('diabetes_prediction_dataset.csv')
clean_data = clean_and_encode(df, df)['Dataset_1']
X_train, y_train = clean_data['X_train'], clean_data['y_train']
X_val, y_val = clean_data['X_val'], clean_data['y_val']
X_test, y_test = clean_data['X_test'], clean_data['y_test']


base_models = [
    ('rf', RandomForestClassifier(random_state=42)),
    ('gb', GradientBoostingClassifier(random_state=42)),
    ('svm', SVC(probability=True, random_state=42)), # Phải bật probability=True cho Stacking
    ('xgb', XGBClassifier(use_label_encoder=False, eval_metric='logloss', random_state=42))
]
meta_model = LogisticRegression()

paper_stacking_ensemble = StackingClassifier(
    estimators=base_models,
    final_estimator=meta_model,
    cv=5 
)

paper_stacking_ensemble.fit(X_train, y_train)
y_pred = paper_stacking_ensemble.predict(X_test)

print("Kết quả phương pháp Stacking của Bài báo tham chiếu:")
print(f"Accuracy: {accuracy_score(y_test, y_pred):.4f}")
print(f"F1-Score: {f1_score(y_test, y_pred):.4f}")

Kết quả phương pháp Stacking của Bài báo tham chiếu:
Accuracy: 0.9703
F1-Score: 0.8118
